# Phase 4: all diagnoses, H3/H4/H5, attackers, Siamese comparator

Run on Google Colab with Google Drive mounted. Each phase starts only if the previous phase's gate file says GO. See the repository README for the full workflow.

### Cell 4.0: Start the session; verify the frozen model

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import os, sys, json, shutil, importlib, time, math, hashlib
from pathlib import Path
import numpy as np, pandas as pd, torch

PROJECT = Path("/content/drive/MyDrive/ecg_leak")
sys.path.insert(0, str(PROJECT / "src"))
import ecgleak.config as ecfg
importlib.reload(ecfg)
CFG = ecfg.load(str(PROJECT / "config.yaml"))
PATHS = CFG["paths"]
gate3 = json.load(open(f"{PATHS['gates']}/gate_3.json"))
ready = json.load(open(f"{PATHS['gates']}/gate_3_ready.json"))
PLAN = json.load(open(f"{PATHS['gates']}/phase3_plan.json"))
assert gate3["go"], "Phase 3 gate is not GO"
MAIN_DIR = f"{PATHS['runs']}/id_main_s0"
assert hashlib.sha256(open(f"{MAIN_DIR}/best.pt", "rb").read()).hexdigest() == ready["model_sha256"], \
    "The identity model on Drive is not the one frozen in Phase 3"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
WINDOW = int(CFG["preprocess"]["window_s"] * CFG["data"]["fs"])
OUT = f"{PATHS['results']}/phase4"
os.makedirs(OUT, exist_ok=True)

LOCAL_CACHE = "/content/cache"
os.makedirs(LOCAL_CACHE, exist_ok=True)
splits = pd.read_csv(f"{PATHS['splits']}/splits.csv", index_col="ecg_id")
def load_split(s):
    dst = f"{LOCAL_CACHE}/{s}.npz"
    if not os.path.isfile(dst):
        shutil.copyfile(f"{PATHS['cache']}/{s}.npz", dst)
    z = np.load(dst, allow_pickle=False)
    d = {k: z[k] for k in z.files}
    d["date"] = pd.to_datetime(splits.loc[d["ecg_id"], "recording_date"]).to_numpy()
    return d
D = {s: load_split(s) for s in ["id_train", "id_eval", "probe_dev", "lockbox"]}
SUPER = list(D["probe_dev"]["label_names"])
print("Phase 3: GO | frozen model verified | H1", "SUPPORTED" if gate3["H1_supported"] else "NOT SUPPORTED",
      "| Device:", DEVICE)

### Cell 4.1: Updated training code: train.py

In [ ]:
%%writefile /content/drive/MyDrive/ecg_leak/src/ecgleak/train.py
import glob, math, os, time
import numpy as np, torch, torch.nn as nn, torch.nn.functional as F
from .models import build_encoder, ArcFaceHead

def _schedule(opt, total, warm):
    def f(step):
        if step < warm:
            return (step + 1) / max(1, warm)
        return 0.5 * (1 + math.cos(math.pi * min(1.0, (step - warm) / max(1, total - warm))))
    return torch.optim.lr_scheduler.LambdaLR(opt, f)

def latest_checkpoint(run_dir):
    files = sorted(glob.glob(os.path.join(run_dir, "checkpoints", "epoch_*.pt")))
    return files[-1] if files else None

def _crops(signals, idx, offs, scale, window, rng):
    xb = np.stack([signals[i, o:o + window] for i, o in zip(idx, offs)]).astype(np.float32)
    return (xb * scale[:, None] + rng.normal(0, 0.05, xb.shape)).astype(np.float32)

def train_identity(cfg, signals, patient_ids, run_dir, device, epochs=None, eval_fn=None, crops_per_record=4,
                   seed=None, stop_after=None, max_steps_per_epoch=None, log=print, loss="arcface",
                   patience=None, snapshot_epochs=(), margin=0.3):
    """Identity training (ArcFace, or a Siamese contrastive loss) with automatic resume from run_dir."""
    t, seed = cfg["train"], cfg["seed"] if seed is None else seed
    epochs, patience = epochs or t["epochs"], patience or t["patience"]
    window = int(cfg["preprocess"]["window_s"] * cfg["data"]["fs"])
    classes, y_all = np.unique(patient_ids, return_inverse=True)
    members = [np.where(y_all == c)[0] for c in range(len(classes))]
    n, L, B = len(signals), signals.shape[1], t["batch"]
    steps = math.ceil(n * crops_per_record / B)
    if max_steps_per_epoch:
        steps = min(steps, max_steps_per_epoch)

    torch.manual_seed(seed)
    model = build_encoder(cfg).to(device)
    head = ArcFaceHead(cfg["model"]["emb_dim"], len(classes), cfg["arcface"]["s"], cfg["arcface"]["m"]).to(device) \
        if loss == "arcface" else None
    params = list(model.parameters()) + (list(head.parameters()) if head is not None else [])
    opt = torch.optim.AdamW(params, lr=t["lr"], weight_decay=t["weight_decay"])
    sched = _schedule(opt, steps * epochs, int(t["warmup_frac"] * steps * epochs))
    scaler = torch.amp.GradScaler("cuda", enabled=(device == "cuda"))
    os.makedirs(os.path.join(run_dir, "checkpoints"), exist_ok=True)
    os.makedirs(os.path.join(run_dir, "snapshots"), exist_ok=True)

    start, history, best, bad = 0, [], float("inf"), 0
    ck = latest_checkpoint(run_dir)
    if ck:
        st = torch.load(ck, map_location=device, weights_only=False)
        model.load_state_dict(st["model"])
        if head is not None:
            head.load_state_dict(st["head"])
        opt.load_state_dict(st["opt"]); sched.load_state_dict(st["sched"]); scaler.load_state_dict(st["scaler"])
        torch.set_rng_state(st["torch_rng"].cpu())
        if device == "cuda" and st.get("cuda_rng") is not None:
            torch.cuda.set_rng_state_all([s.cpu() for s in st["cuda_rng"]])
        start, history, best, bad = st["epoch"] + 1, st["history"], st["best"], st["bad"]
        log(f"Resumed from {os.path.basename(ck)} -> continuing at epoch {start + 1}")
        if bad >= patience or (stop_after is not None and start >= stop_after):
            log("This run had already finished.")
            return model, history

    for epoch in range(start, epochs):
        t0 = time.time()
        model.train()
        if head is not None:
            head.train()
        rng = np.random.default_rng(seed * 1000 + epoch)
        idx = np.repeat(np.arange(n), crops_per_record); rng.shuffle(idx)
        tot_loss = tot_acc = 0.0
        for b in range(steps):
            ii = idx[b * B:(b + 1) * B]
            m = len(ii)
            if loss == "arcface":
                xb = _crops(signals, ii, rng.integers(0, L - window + 1, m), rng.uniform(0.9, 1.1, m), window, rng)
                yb = torch.from_numpy(y_all[ii]).long().to(device)
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
                    emb = model(torch.from_numpy(xb).to(device))
                logits = head(emb.float(), yb)
                L_b = F.cross_entropy(logits, yb)
                acc = (logits.argmax(1) == yb).float().mean().item()
            else:
                pos = np.array([rng.choice(members[y_all[i]][members[y_all[i]] != i])
                                if len(members[y_all[i]]) > 1 else i for i in ii])
                neg = rng.integers(0, n, m)
                clash = y_all[neg] == y_all[ii]
                while clash.any():
                    neg[clash] = rng.integers(0, n, clash.sum()); clash = y_all[neg] == y_all[ii]
                allx = np.concatenate([_crops(signals, j, rng.integers(0, L - window + 1, m), rng.uniform(0.9, 1.1, m), window, rng)
                                       for j in (ii, pos, neg)])
                with torch.autocast(device_type="cuda", dtype=torch.float16, enabled=(device == "cuda")):
                    e = model(torch.from_numpy(allx).to(device)).float()
                a, p, q = e[:m], e[m:2 * m], e[2 * m:]
                cap, can = (a * p).sum(1), (a * q).sum(1)
                L_b = ((1 - cap) ** 2).mean() + (F.relu(can - margin) ** 2).mean()
                acc = (cap > can).float().mean().item()
            opt.zero_grad(set_to_none=True)
            scaler.scale(L_b).backward()
            scaler.unscale_(opt)
            torch.nn.utils.clip_grad_norm_(params, 5.0)
            scaler.step(opt); scaler.update(); sched.step()
            tot_loss += L_b.item(); tot_acc += acc
        rec = {"epoch": epoch + 1, "loss": tot_loss / steps, "train_acc": tot_acc / steps,
               "lr": sched.get_last_lr()[0], "seconds": round(time.time() - t0, 1)}
        if eval_fn is not None:
            rec.update(eval_fn(model))
        if rec.get("eer", -epoch) < best:
            best, bad = rec.get("eer", -epoch), 0
            torch.save({"model": model.state_dict(), "epoch": epoch + 1, "eer": rec.get("eer")},
                       os.path.join(run_dir, "best.pt"))
        else:
            bad += 1
        if epoch + 1 in snapshot_epochs:
            torch.save({"model": model.state_dict(), "epoch": epoch + 1, "eer": rec.get("eer")},
                       os.path.join(run_dir, "snapshots", f"epoch_{epoch + 1:03d}.pt"))
        history.append(rec)
        path = os.path.join(run_dir, "checkpoints", f"epoch_{epoch + 1:03d}.pt")
        torch.save({"epoch": epoch, "model": model.state_dict(), "head": head.state_dict() if head is not None else None,
                    "opt": opt.state_dict(), "sched": sched.state_dict(), "scaler": scaler.state_dict(),
                    "history": history, "best": best, "bad": bad, "torch_rng": torch.get_rng_state(),
                    "cuda_rng": torch.cuda.get_rng_state_all() if device == "cuda" else None}, path + ".tmp")
        os.replace(path + ".tmp", path)
        for old in sorted(glob.glob(os.path.join(run_dir, "checkpoints", "epoch_*.pt")))[:-3]:
            os.remove(old)
        log(" | ".join(f"{k} {v:.4f}" if isinstance(v, float) else f"{k} {v}" for k, v in rec.items()))
        if stop_after is not None and epoch + 1 >= stop_after:
            break
        if bad >= patience:
            log("Early stopping: identity EER stopped improving")
            break
    return model, history

def load_best(cfg, run_dir, device, name="best.pt"):
    model = build_encoder(cfg).to(device)
    st = torch.load(os.path.join(run_dir, name), map_location=device, weights_only=False)
    model.load_state_dict(st["model"])
    return model.eval(), st

class DiagNet(nn.Module):
    """Same encoder + one linear output: a supervised diagnosis model (used only for attributions, H4)."""
    def __init__(self, encoder):
        super().__init__()
        self.encoder, self.out = encoder, nn.Linear(encoder.emb_dim, 1)
    def forward(self, x):
        return self.out(self.encoder(x)).squeeze(1)

def train_diagnosis(cfg, signals, y, path, device, epochs=15, seed=0, log=print):
    window = int(cfg["preprocess"]["window_s"] * cfg["data"]["fs"])
    torch.manual_seed(seed)
    net = DiagNet(build_encoder(cfg)).to(device)
    if os.path.isfile(path):
        net.load_state_dict(torch.load(path, map_location=device, weights_only=False))
        log(f"Loaded existing {os.path.basename(path)}")
        return net.eval()
    opt = torch.optim.AdamW(net.parameters(), lr=1e-3, weight_decay=1e-4)
    pw = torch.tensor([(1 - y).sum() / max(1, y.sum())], device=device, dtype=torch.float32)
    n, L, B = len(signals), signals.shape[1], cfg["train"]["batch"]
    for epoch in range(epochs):
        net.train()
        rng = np.random.default_rng(seed * 1000 + epoch)
        idx = np.repeat(np.arange(n), 2); rng.shuffle(idx)
        tot = 0.0
        for b in range(math.ceil(len(idx) / B)):
            ii = idx[b * B:(b + 1) * B]
            xb = _crops(signals, ii, rng.integers(0, L - window + 1, len(ii)), rng.uniform(0.9, 1.1, len(ii)), window, rng)
            logit = net(torch.from_numpy(xb).to(device))
            l = F.binary_cross_entropy_with_logits(logit, torch.from_numpy(y[ii]).float().to(device), pos_weight=pw)
            opt.zero_grad(set_to_none=True); l.backward(); opt.step(); tot += l.item()
        log(f"  diag epoch {epoch + 1}: loss {tot / math.ceil(len(idx) / B):.4f}")
    torch.save(net.state_dict(), path)
    return net.eval()

### Cell 4.2: Analysis code: analysis.py

In [ ]:
%%writefile /content/drive/MyDrive/ecg_leak/src/ecgleak/analysis.py
import warnings
import numpy as np, pandas as pd, torch, torch.nn.functional as F
from scipy.stats import spearmanr
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from . import leakage as LK
warnings.filterwarnings("ignore", category=FutureWarning, module="sklearn")

def fit_and_score(X_dev, y_dev, folds, X_test, C_grid, attacker="logreg", pca=None, seed=0):
    """Same protocol as Phase 3, with a choice of attacker."""
    if attacker == "logreg":
        return LK.fit_and_score(X_dev, y_dev, folds, X_test, C_grid, pca=pca, seed=seed)[0]
    cv = [(np.where(folds != f)[0], np.where(folds == f)[0]) for f in np.unique(folds)]
    if attacker == "mlp":
        model = make_pipeline(StandardScaler(), MLPClassifier((256,), alpha=1e-3, early_stopping=True,
                                                              max_iter=300, random_state=seed))
        model.fit(X_dev, y_dev)
    elif attacker == "knn":
        model = GridSearchCV(make_pipeline(StandardScaler(), KNeighborsClassifier(weights="distance")),
                             {"kneighborsclassifier__n_neighbors": [5, 15, 31, 63]}, cv=cv, scoring="roc_auc")
        model.fit(X_dev, y_dev)
    else:
        raise ValueError(attacker)
    return model.predict_proba(X_test)[:, 1]

def gains(y, scores, n_boot, n_perm, seed=0):
    """Delta_rand / Delta_raw / Delta_demo with paired-bootstrap 95% CIs and permutation p-values (as Phase 3)."""
    aucs = {k: LK.auc(y, s) for k, s in scores.items()}
    boot = LK.paired_bootstrap(y, scores, n_boot, seed)
    untr = [k for k in scores if k.startswith("untrained_")]
    rows = []
    def add(name, est, b, a, bb):
        lo, hi = np.percentile(b, [2.5, 97.5])
        rows.append({"gain": name, "estimate": est, "ci_low": lo, "ci_high": hi, "ci_halfwidth": (hi - lo) / 2,
                     "p_perm": LK.paired_permutation(y, a, bb, n_perm, seed)})
    pooled = np.mean([LK.percentile_rank(scores[k]) for k in untr], axis=0)
    add("Delta_rand", aucs["identity_template"] - np.mean([aucs[k] for k in untr]),
        boot["identity_template"] - np.mean([boot[k] for k in untr], axis=0), scores["identity_template"], pooled)
    if "raw_median_beat" in scores:
        add("Delta_raw", aucs["identity_template"] - aucs["raw_median_beat"],
            boot["identity_template"] - boot["raw_median_beat"], scores["identity_template"], scores["raw_median_beat"])
    if "age_sex" in scores:
        add("Delta_demo", aucs["template_plus_age_sex"] - aucs["age_sex"],
            boot["template_plus_age_sex"] - boot["age_sex"], scores["template_plus_age_sex"], scores["age_sex"])
    return pd.DataFrame(rows).set_index("gain"), aucs

def bh(p):
    """Benjamini-Hochberg adjusted p-values."""
    p = np.asarray(p, dtype=float); n = len(p); o = np.argsort(p)
    q = p[o] * n / np.arange(1, n + 1)
    q = np.minimum.accumulate(q[::-1])[::-1]
    out = np.empty(n); out[o] = np.minimum(q, 1)
    return out

def integrated_gradients(f, x, steps=32):
    """IG from a zero baseline; f maps a batch of windows to one number per window."""
    total = torch.zeros_like(x)
    for a in torch.linspace(1.0 / steps, 1.0, steps, device=x.device):
        xi = (a * x).detach().requires_grad_(True)
        g, = torch.autograd.grad(f(xi).sum(), xi)
        total += g
    return (x * total / steps).detach()

def identity_score_fn(model, x_full_windows):
    """Output to explain for an identity encoder: similarity of window 1 to the record's own template."""
    with torch.no_grad():
        ref = F.normalize(model(x_full_windows.reshape(-1, x_full_windows.shape[-1]))
                          .reshape(x_full_windows.shape[0], x_full_windows.shape[1], -1).mean(1), dim=1)
    return lambda xi: (model(xi) * ref).sum(1)

def overlap(a, b):
    """Per-record Spearman rank correlation of |attribution| over time (raw values, no min-max offset)."""
    return np.array([spearmanr(np.abs(u), np.abs(v))[0] for u, v in zip(a, b)])

### Cell 4.3: Build every representation once

In [ ]:
import ecgleak.models as M, ecgleak.train as TR, ecgleak.evaluate as EV, ecgleak.leakage as LK, ecgleak.analysis as AN
for mod in (M, TR, EV, LK, AN):
    importlib.reload(mod)
id_model, _ = TR.load_best(CFG, MAIN_DIR, DEVICE)
UNTRAINED = M.untrained_copies(CFG, PLAN["untrained_seeds"], DEVICE)

def all_reps(d, enc=None):
    enc = enc or id_model
    r = {"identity_template": EV.embed(enc, d["signal"], WINDOW, DEVICE)}
    for k, m in enumerate(UNTRAINED):
        r[f"untrained_{k}"] = EV.embed(m, d["signal"], WINDOW, DEVICE)
    r["raw_median_beat"] = d["median_beat"]
    r["age_sex"] = EV.demo_features(d["age"], d["sex"])
    r["template_plus_age_sex"] = np.c_[r["identity_template"], r["age_sex"]]
    return r
REP = {s: all_reps(D[s]) for s in ["probe_dev", "lockbox"]}
pdv, idv = D["probe_dev"], D["id_eval"]
def id_eval_fn(model):
    r = EV.verification(EV.embed(model, idv["signal"], WINDOW, DEVICE), idv["patient_id"], idv["date"])
    return {"eer": r["eer"], "rank1": r["rank1"]}
print("Representations ready:", list(REP["lockbox"]))

def run_task(cls, reps=REP, names=None, attacker="logreg"):
    keep_d, y_d = EV.task_data(D["probe_dev"]["labels"], SUPER, cls)
    keep_l, y_l = EV.task_data(D["lockbox"]["labels"], SUPER, cls)
    folds = D["probe_dev"]["fold"][keep_d]
    names = names or list(reps["lockbox"])
    sc = {k: AN.fit_and_score(reps["probe_dev"][k][keep_d], y_d, folds, reps["lockbox"][k][keep_l], PLAN["C_grid"],
                              attacker=attacker, pca=64 if k == "raw_median_beat" else None, seed=CFG["seed"])
          for k in names}
    return y_l, sc

### Cell 4.4: All four diagnoses with BH correction

In [ ]:
tables, auc_rows = [], []
for cls in CFG["leakage"]["tasks"]:
    y_l, sc = run_task(cls)
    if y_l.sum() < 30 or (1 - y_l).sum() < 30:
        print(f"{cls}: too few lockbox records, skipped"); continue
    tab, aucs = AN.gains(y_l, sc, PLAN["n_boot"], PLAN["n_perm"], CFG["seed"])
    tables.append(tab.assign(task=f"{cls} vs NORM", n_pos=int(y_l.sum()), n_neg=int((1 - y_l).sum())).reset_index())
    auc_rows.append({"task": f"{cls} vs NORM", **aucs})
    print(f"{cls} done")
sec = pd.concat(tables, ignore_index=True)
sec["p_BH"] = np.nan
for g, idx in sec.groupby("gain").groups.items():
    sec.loc[idx, "p_BH"] = AN.bh(sec.loc[idx, "p_perm"])
sec = sec[["task", "gain", "estimate", "ci_low", "ci_high", "p_perm", "p_BH", "n_pos", "n_neg"]]
sec.to_csv(f"{OUT}/table3_all_tasks.csv", index=False)
pd.DataFrame(auc_rows).to_csv(f"{OUT}/aucs_all_tasks.csv", index=False)
print(sec.round(4).to_string(index=False))
cd = sec[(sec.task.str.startswith(CFG["leakage"]["primary"] + " ")) & (sec.gain == "Delta_rand")]["estimate"]
p3 = [g["estimate"] for g in gate3["gains"] if g["gain"] == "Delta_rand"][0]
CONSISTENT = len(cd) == 1 and abs(cd.iloc[0] - p3) < 1e-3
print(f"\nConsistency check: CD Delta_rand here {cd.iloc[0]:+.4f} vs Phase 3 {p3:+.4f} ->", "PASS" if CONSISTENT else "FAIL")

### Cell 4.5: H5: persistence vs leakage

In [ ]:
from scipy.stats import spearmanr
persist = pd.read_csv(f"{PATHS['interim']}/persistence.csv", index_col=0)
dr = sec[sec.gain == "Delta_rand"].assign(cls=lambda t: t.task.str.split(" ").str[0]).set_index("cls")
h5 = dr[["estimate", "ci_low", "ci_high"]].join(persist[["persistence", "visit-1 positives"]], how="inner")
h5 = h5.sort_values("persistence", ascending=False)
rho, p = spearmanr(h5["persistence"], h5["estimate"]) if len(h5) >= 3 else (np.nan, np.nan)
h5.to_csv(f"{OUT}/h5_persistence_vs_leak.csv")
print(h5.round(3).to_string())
print(f"\nSpearman rho (persistence vs Delta_rand) = {rho:+.2f} over {len(h5)} diagnoses (p = {p:.2f}); "
      "with 4 points this is descriptive only.")

import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(5.5, 3.5))
ax.errorbar(h5["persistence"], h5["estimate"], yerr=[h5["estimate"] - h5["ci_low"], h5["ci_high"] - h5["estimate"]],
            fmt="o", capsize=4, color="black")
for c, r in h5.iterrows():
    ax.annotate(c, (r["persistence"], r["estimate"]), xytext=(6, 4), textcoords="offset points")
ax.axhline(0, ls="--", c="grey"); ax.set_xlabel("Persistence across visits (P[still present at next visit])")
ax.set_ylabel("Delta_rand (lockbox)"); ax.set_title("H5: do persistent diagnoses leak more?")
plt.tight_layout(); plt.savefig(f"{PATHS['figures']}/figure2_persistence_vs_leak.png", dpi=200); plt.show()

### Cell 4.6: Stronger attackers (MLP, kNN)

In [ ]:
att_rows = []
names = ["identity_template"] + [f"untrained_{k}" for k in range(len(UNTRAINED))]
for attacker in ["logreg", "mlp", "knn"]:
    y_l, sc = run_task(CFG["leakage"]["primary"], names=names, attacker=attacker)
    tab, aucs = AN.gains(y_l, sc, PLAN["n_boot"], PLAN["n_perm"], CFG["seed"])
    r = tab.loc["Delta_rand"]
    att_rows.append({"attacker": attacker, "auroc_template": aucs["identity_template"],
                     "auroc_untrained_mean": np.mean([aucs[k] for k in names[1:]]),
                     "Delta_rand": r["estimate"], "ci_low": r["ci_low"], "ci_high": r["ci_high"], "p_perm": r["p_perm"]})
    print(f"{attacker} done")
att = pd.DataFrame(att_rows).set_index("attacker")
att.to_csv(f"{OUT}/attackers_{CFG['leakage']['primary']}.csv")
print(att.round(4).to_string())

### Cell 4.7: H3: utility-leakage coupling

In [ ]:
import matplotlib.pyplot as plt
from scipy.stats import spearmanr
COUP_DIR = f"{PATHS['runs']}/coupling_s0"
main_epochs = len(pd.read_csv(f"{MAIN_DIR}/history.csv"))          # 19 in your main run
SNAPS = tuple(sorted({1, 2, 3, 4, 5, 6, 8, 10, 12, 15, main_epochs} & set(range(1, main_epochs + 1))))
ecfg.freeze(CFG, COUP_DIR)
TR.train_identity(CFG, D["id_train"]["signal"], D["id_train"]["patient_id"], COUP_DIR, DEVICE,
                  stop_after=main_epochs, eval_fn=id_eval_fn, patience=10 ** 6, snapshot_epochs=SNAPS)

keep, y = EV.task_data(pdv["labels"], SUPER, CFG["leakage"]["primary"])
folds, sig = pdv["fold"][keep], pdv["signal"][keep]
rand_dev = np.mean([EV.cv_auroc(REP["probe_dev"][f"untrained_{k}"][keep], y, folds, PLAN["C_grid"])[0]
                    for k in range(len(UNTRAINED))])
rows = []
for f in sorted(os.listdir(f"{COUP_DIR}/snapshots")):
    m, st = TR.load_best(CFG, COUP_DIR, DEVICE, name=f"snapshots/{f}")
    a = EV.cv_auroc(EV.embed(m, sig, WINDOW, DEVICE), y, folds, PLAN["C_grid"])[0]
    rows.append({"epoch": st["epoch"], "eer": st["eer"], "auroc_dev": a, "Delta_rand_dev": a - rand_dev})
    print(f"epoch {st['epoch']:3d}: EER {st['eer']:.3f} | Delta_rand (probe-dev) {a - rand_dev:+.3f}")
coup = pd.DataFrame(rows)
coup.to_csv(f"{OUT}/h3_coupling.csv", index=False)
rho3, p3c = spearmanr(coup["eer"], coup["Delta_rand_dev"])
print(f"\nH3: Spearman rho(EER, Delta_rand) = {rho3:+.2f} (p = {p3c:.3f}) over {len(coup)} checkpoints. "
      "Negative = a better biometric (lower EER) leaks more.")
fig, ax = plt.subplots(figsize=(5.5, 3.5))
ax.plot(coup["eer"], coup["Delta_rand_dev"], "o-", color="black")
for _, r in coup.iterrows():
    ax.annotate(int(r["epoch"]), (r["eer"], r["Delta_rand_dev"]), xytext=(4, 4), textcoords="offset points", fontsize=8)
ax.invert_xaxis(); ax.set_xlabel("Identity EER on ID-eval (better to the right)"); ax.set_ylabel("Delta_rand (probe-dev)")
ax.set_title("H3: does a better biometric leak more?")
plt.tight_layout(); plt.savefig(f"{PATHS['figures']}/figure3_coupling.png", dpi=200); plt.show()

### Cell 4.8: BiomECG-style Siamese comparator

In [ ]:
SIAM_DIR = f"{PATHS['runs']}/siamese_s0"
ecfg.freeze(CFG, SIAM_DIR)
TR.train_identity(CFG, D["id_train"]["signal"], D["id_train"]["patient_id"], SIAM_DIR, DEVICE,
                  eval_fn=id_eval_fn, loss="contrastive")
siam, st_s = TR.load_best(CFG, SIAM_DIR, DEVICE)
siam_id = EV.verification(EV.embed(siam, idv["signal"], WINDOW, DEVICE), idv["patient_id"], idv["date"])
names = ["identity_template"] + [f"untrained_{k}" for k in range(len(UNTRAINED))]
REP_S = {s: {**{k: REP[s][k] for k in names[1:]}, "identity_template": EV.embed(siam, D[s]["signal"], WINDOW, DEVICE)}
         for s in ["probe_dev", "lockbox"]}
y_l, sc = run_task(CFG["leakage"]["primary"], reps=REP_S, names=names)
tab_s, aucs_s = AN.gains(y_l, sc, PLAN["n_boot"], PLAN["n_perm"], CFG["seed"])
arc = [g for g in gate3["gains"] if g["gain"] == "Delta_rand"][0]
comp = pd.DataFrame([
    {"model": "ArcFace (main)", "eer": gate3["identity"]["eer"], "rank1": gate3["identity"]["rank1"],
     "Delta_rand": arc["estimate"], "ci_low": arc["ci_low"], "ci_high": arc["ci_high"]},
    {"model": "Siamese contrastive (BiomECG-style)", "eer": siam_id["eer"], "rank1": siam_id["rank1"],
     "Delta_rand": tab_s.loc["Delta_rand", "estimate"], "ci_low": tab_s.loc["Delta_rand", "ci_low"],
     "ci_high": tab_s.loc["Delta_rand", "ci_high"]}]).set_index("model")
comp.to_csv(f"{OUT}/siamese_vs_arcface.csv")
print(comp.round(4).to_string())

### Cell 4.9: H4: repaired attribution-overlap check

In [ ]:
from ecgleak.analysis import integrated_gradients, identity_score_fn, overlap
keep_d, y_d = EV.task_data(pdv["labels"], SUPER, CFG["leakage"]["primary"])
diag = TR.train_diagnosis(CFG, pdv["signal"][keep_d], y_d, f"{PATHS['runs']}/diag_{CFG['leakage']['primary']}.pt", DEVICE)
diag_shuf = TR.train_diagnosis(CFG, pdv["signal"][keep_d], np.random.default_rng(7).permutation(y_d),
                               f"{PATHS['runs']}/diag_{CFG['leakage']['primary']}_shuffled.pt", DEVICE, seed=1)

LB = D["lockbox"]
keep_l, _ = EV.task_data(LB["labels"], SUPER, CFG["leakage"]["primary"])
pick = np.random.default_rng(CFG["seed"]).choice(np.where(keep_l)[0], size=min(300, keep_l.sum()), replace=False)
full = torch.from_numpy(LB["signal"][pick, :4 * WINDOW].astype(np.float32)).to(DEVICE).reshape(len(pick), 4, WINDOW)
x0 = full[:, 0, :]

def attributions(f, bs=32):
    return torch.cat([integrated_gradients(f, x0[i:i + bs]) for i in range(0, len(x0), bs)]).cpu().numpy()

def id_attr(model):
    outs = []
    for i in range(0, len(x0), 32):
        f = identity_score_fn(model, full[i:i + 32])
        outs.append(integrated_gradients(f, x0[i:i + 32]))
    return torch.cat(outs).cpu().numpy()

A_id = id_attr(id_model)
A_untr = id_attr(UNTRAINED[0])
A_diag = attributions(lambda x: diag(x))
A_shuf = attributions(lambda x: diag_shuf(x))
ov = pd.DataFrame({"identity_vs_diagnosis": overlap(A_id, A_diag),
                   "null_identity_vs_shuffled_diagnosis": overlap(A_id, A_shuf),
                   "null_untrained_vs_diagnosis": overlap(A_untr, A_diag)})
ov.to_csv(f"{OUT}/h4_attribution_overlap.csv", index=False)
rng = np.random.default_rng(CFG["seed"])
bi = rng.integers(0, len(ov), (PLAN["n_boot"], len(ov)))
h4 = {}
for null in ["null_identity_vs_shuffled_diagnosis", "null_untrained_vs_diagnosis"]:
    d = ov["identity_vs_diagnosis"].to_numpy() - ov[null].to_numpy()
    b = d[bi].mean(1)
    h4[null] = {"mean_difference": float(d.mean()), "ci_low": float(np.percentile(b, 2.5)), "ci_high": float(np.percentile(b, 97.5))}
print(ov.mean().round(3).to_string())
print(pd.DataFrame(h4).T.round(3).to_string())
H4 = all(v["ci_low"] > 0 for v in h4.values())
print("H4 (identity and diagnosis attributions overlap beyond both nulls):", "SUPPORTED" if H4 else "NOT SUPPORTED")

### Cell 4.10: Phase 4 gate

In [ ]:
dr = sec[sec.gain == "Delta_rand"].set_index("task")
H3 = bool(rho3 < 0 and p3c < 0.05)
summary = {
    "all_tasks_Delta_rand": dr[["estimate", "ci_low", "ci_high", "p_BH"]].round(4).reset_index().to_dict("records"),
    "tasks_with_significant_leak_after_BH": dr.index[(dr["p_BH"] < 0.05) & (dr["ci_low"] > 0)].tolist(),
    "H5_spearman_persistence_vs_leak": rho, "H3_spearman_eer_vs_leak": rho3, "H3_p": p3c, "H3_supported": H3,
    "attackers": att.round(4).reset_index().to_dict("records"),
    "siamese": comp.round(4).reset_index().to_dict("records"), "H4_supported": H4, "H4_detail": h4,
}
checks = {"CD result matches Phase 3 (same pipeline)": CONSISTENT, "All four tasks scored": len(dr) == len(CFG["leakage"]["tasks"]),
          "H5 table written": os.path.isfile(f"{OUT}/h5_persistence_vs_leak.csv"),
          "H3 coupling run complete": len(coup) >= 3, "Stronger attackers run": len(att) == 3,
          "Siamese comparator run": len(comp) == 2, "H4 attribution check run": len(ov) > 0}
GO = all(checks.values())
print("PHASE 4 GATE\n" + "-" * 55)
for k, v in checks.items():
    print(f"  {'PASS' if v else 'FAIL'}  {k}")
print("-" * 55)
print("Diseases with a significant leak after BH:", summary["tasks_with_significant_leak_after_BH"] or "none")
print(f"H3 (better biometric leaks more): {'SUPPORTED' if H3 else 'NOT SUPPORTED'} (rho {rho3:+.2f}, p {p3c:.3f})")
print(f"H4 (attribution overlap is real): {'SUPPORTED' if H4 else 'NOT SUPPORTED'}")
print(f"H5 (persistent diagnoses leak more): rho {rho:+.2f}, descriptive only")
json.dump({"phase": 4, "go": GO, "checks": checks, **summary, "environment": ecfg.environment()},
          open(f"{PATHS['gates']}/gate_4.json", "w"), indent=2, default=str)
print("GO: next is Phase 5 (robustness: seeds, lead II, external data)." if GO else "NO-GO: fix the FAIL lines above.")